# Options Volatility Surface and Greeks Dashboard

This project analyzes SPY options by calculating implied volatility and Greeks, then visualizing volatility, liquidity, and option sensitivities across strikes and expiration dates.


## 0. Setup

Install the required packages and import the libraries used in the notebook.


In [1]:
%pip install -q yfinance plotly scipy pandas numpy nbformat


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.interpolate import griddata
from scipy.optimize import brentq
from scipy.stats import norm

from IPython.display import display

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    import yfinance as yf
except ImportError:
    yf = None

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)

## 1. Settings

Main assumptions and filters used in the analysis.


In [3]:
TICKER = "SPY"
RISK_FREE_RATE = 0.043       # 4.3% assumption
MIN_DTE = 7                 # minimum days to expiration
MAX_DTE = 365               # maximum days to expiration
MAX_EXPIRATIONS = 9
MIN_MONEYNESS = 0.70        # strike / spot
MAX_MONEYNESS = 1.30
MAX_RELATIVE_SPREAD = 0.60
MAX_QUOTE_AGE_DAYS = 21
CONTRACT_MULTIPLIER = 100
OUTPUT_DIR = Path("options_dashboard_outputs")

## 2. Black-Scholes, implied volatility, and Greeks

Black-Scholes calculates a theoretical option price when volatility and the other inputs are known. Since market prices are observable, implied volatility is found by solving for the volatility that makes the model price match the market price.


In [4]:
def bsm_price(option_type, spot, strike, time_to_expiry, rate, dividend_yield, volatility):
    """Black-Scholes-Merton price for a European call or put."""
    if time_to_expiry <= 0:
        if option_type == "call":
            return max(spot - strike, 0.0)
        return max(strike - spot, 0.0)

    if spot <= 0 or strike <= 0 or volatility <= 0:
        return np.nan

    sqrt_t = math.sqrt(time_to_expiry)
    d1 = (
        math.log(spot / strike)
        + (rate - dividend_yield + 0.5 * volatility**2) * time_to_expiry
    ) / (volatility * sqrt_t)
    d2 = d1 - volatility * sqrt_t

    discounted_spot = spot * math.exp(-dividend_yield * time_to_expiry)
    discounted_strike = strike * math.exp(-rate * time_to_expiry)

    if option_type == "call":
        return discounted_spot * norm.cdf(d1) - discounted_strike * norm.cdf(d2)
    return discounted_strike * norm.cdf(-d2) - discounted_spot * norm.cdf(-d1)


def option_price_bounds(option_type, spot, strike, time_to_expiry, rate, dividend_yield):
    """Simple no-arbitrage lower and upper price bounds."""
    discounted_spot = spot * math.exp(-dividend_yield * time_to_expiry)
    discounted_strike = strike * math.exp(-rate * time_to_expiry)

    if option_type == "call":
        return max(discounted_spot - discounted_strike, 0.0), discounted_spot
    return max(discounted_strike - discounted_spot, 0.0), discounted_strike


def solve_iv(option_type, market_price, spot, strike, time_to_expiry, rate, dividend_yield):
    """Find the volatility that makes Black-Scholes equal the market price."""
    if market_price <= 0 or time_to_expiry <= 0:
        return np.nan

    lower_price, upper_price = option_price_bounds(
        option_type, spot, strike, time_to_expiry, rate, dividend_yield
    )
    if market_price < lower_price - 0.02 or market_price > upper_price + 0.02:
        return np.nan

    def pricing_error(volatility):
        return bsm_price(
            option_type, spot, strike, time_to_expiry,
            rate, dividend_yield, volatility
        ) - market_price

    try:
        return brentq(pricing_error, 0.0001, 5.0)
    except (ValueError, RuntimeError):
        return np.nan


def bsm_greeks(option_type, spot, strike, time_to_expiry, rate, dividend_yield, volatility):
    """Return Delta, Gamma, Vega, Theta, and Rho."""
    if time_to_expiry <= 0 or volatility <= 0:
        return {"delta": np.nan, "gamma": np.nan, "vega": np.nan, "theta": np.nan, "rho": np.nan}

    sqrt_t = math.sqrt(time_to_expiry)
    d1 = (
        math.log(spot / strike)
        + (rate - dividend_yield + 0.5 * volatility**2) * time_to_expiry
    ) / (volatility * sqrt_t)
    d2 = d1 - volatility * sqrt_t

    spot_discount = math.exp(-dividend_yield * time_to_expiry)
    strike_discount = math.exp(-rate * time_to_expiry)
    pdf_d1 = norm.pdf(d1)

    gamma = spot_discount * pdf_d1 / (spot * volatility * sqrt_t)
    vega = spot * spot_discount * pdf_d1 * sqrt_t / 100.0

    if option_type == "call":
        delta = spot_discount * norm.cdf(d1)
        theta_year = (
            -(spot * spot_discount * pdf_d1 * volatility) / (2 * sqrt_t)
            - rate * strike * strike_discount * norm.cdf(d2)
            + dividend_yield * spot * spot_discount * norm.cdf(d1)
        )
        rho = strike * time_to_expiry * strike_discount * norm.cdf(d2) / 100.0
    else:
        delta = -spot_discount * norm.cdf(-d1)
        theta_year = (
            -(spot * spot_discount * pdf_d1 * volatility) / (2 * sqrt_t)
            + rate * strike * strike_discount * norm.cdf(-d2)
            - dividend_yield * spot * spot_discount * norm.cdf(-d1)
        )
        rho = -strike * time_to_expiry * strike_discount * norm.cdf(-d2) / 100.0

    return {
        "delta": delta,
        "gamma": gamma,
        "vega": vega,
        "theta": theta_year / 365,
        "rho": rho,
    }

## 3. Get market data

Pull SPY price history and option-chain data from Yahoo Finance. If live data are unavailable, the notebook uses labeled sample data so the analysis can still run.


In [5]:
def get_live_data(symbol):
    if yf is None:
        raise RuntimeError("yfinance is not installed")

    ticker = yf.Ticker(symbol)
    history = ticker.history(period="2y", auto_adjust=False, actions=True)
    if history.empty:
        raise RuntimeError("No price history returned")

    spot = float(history["Close"].dropna().iloc[-1])
    today = pd.Timestamp.now().normalize().tz_localize(None)

    valid_expirations = []
    for expiration in ticker.options:
        dte = (pd.Timestamp(expiration) - today).days
        if MIN_DTE <= dte <= MAX_DTE:
            valid_expirations.append(expiration)

    selected_expirations = valid_expirations[:MAX_EXPIRATIONS]
    if not selected_expirations:
        raise RuntimeError("No usable option expirations returned")

    frames = []
    for expiration in selected_expirations:
        chain = ticker.option_chain(expiration)

        calls = chain.calls.copy()
        calls["option_type"] = "call"
        calls["expiration"] = pd.Timestamp(expiration)

        puts = chain.puts.copy()
        puts["option_type"] = "put"
        puts["expiration"] = pd.Timestamp(expiration)

        frames.extend([calls, puts])

    raw_options = pd.concat(frames, ignore_index=True)

    annual_dividends = 0.0
    if "Dividends" in history.columns:
        annual_dividends = pd.to_numeric(history["Dividends"], errors="coerce").tail(252).fillna(0).sum()
    dividend_yield = max(float(annual_dividends) / spot, 0.0)

    return {
        "symbol": symbol,
        "spot": spot,
        "history": history,
        "options": raw_options,
        "dividend_yield": dividend_yield,
        "source": "Yahoo Finance live data",
    }

In [6]:
def make_sample_data(symbol):
    """Simple fallback so the notebook still runs when live data are unavailable."""
    rng = np.random.default_rng(42)
    dates = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=504)
    returns = rng.normal(0.00025, 0.011, len(dates))
    close = 500 * np.exp(np.cumsum(returns))

    history = pd.DataFrame({"Close": close, "Dividends": 0.0}, index=dates)
    spot = float(close[-1])
    dividend_yield = 0.013

    rows = []
    today = pd.Timestamp.today().normalize()
    for dte in [30, 60, 90, 150, 210, 300, 360]:
        expiration = today + pd.Timedelta(days=dte)
        time_to_expiry = dte / 365
        forward = spot * math.exp((RISK_FREE_RATE - dividend_yield) * time_to_expiry)

        strikes = np.arange(round(spot * 0.70 / 5) * 5, round(spot * 1.30 / 5) * 5 + 5, 5)
        for strike in strikes:
            log_moneyness = math.log(strike / forward)
            iv = 0.17 - 0.16 * log_moneyness + 0.75 * log_moneyness**2 + 0.025 * math.sqrt(time_to_expiry)
            iv = float(np.clip(iv, 0.08, 0.65))

            for option_type in ["call", "put"]:
                mid = bsm_price(
                    option_type, spot, strike, time_to_expiry,
                    RISK_FREE_RATE, dividend_yield, iv
                )
                spread = max(0.04, mid * 0.03)
                open_interest = int(max(0, 15000 * math.exp(-10 * abs(strike / spot - 1)) * rng.uniform(0.8, 1.2)))

                rows.append({
                    "contractSymbol": f"SAMPLE-{dte}-{option_type}-{strike:.0f}",
                    "lastTradeDate": pd.Timestamp.now(tz="UTC") - pd.Timedelta(hours=4),
                    "strike": strike,
                    "lastPrice": mid,
                    "bid": max(mid - spread / 2, 0),
                    "ask": mid + spread / 2,
                    "volume": int(open_interest * rng.uniform(0.02, 0.12)),
                    "openInterest": open_interest,
                    "impliedVolatility": iv,
                    "option_type": option_type,
                    "expiration": expiration,
                })

    return {
        "symbol": f"{symbol} (sample fallback)",
        "spot": spot,
        "history": history,
        "options": pd.DataFrame(rows),
        "dividend_yield": dividend_yield,
        "source": "Clearly labeled sample data",
    }


try:
    market = get_live_data(TICKER)
except Exception as error:
    print(f"Live data unavailable ({type(error).__name__}). Using sample data instead.")
    market = make_sample_data(TICKER)

print("Source:", market["source"])
print("Symbol:", market["symbol"])
print("Spot price:", round(market["spot"], 2))
print("Raw option rows:", len(market["options"]))

Source: Yahoo Finance live data
Symbol: SPY
Spot price: 767.05
Raw option rows: 2819


## 4. Clean the option chain and calculate IV + Greeks

Create midpoint prices, filter unusable quotes, solve for implied volatility, and calculate the Greeks for each option.


In [7]:
options = market["options"].copy()
spot = market["spot"]
dividend_yield = market["dividend_yield"]
today = pd.Timestamp.now().normalize().tz_localize(None)

for column in ["strike", "bid", "ask", "lastPrice", "volume", "openInterest", "impliedVolatility"]:
    if column not in options.columns:
        options[column] = np.nan
    options[column] = pd.to_numeric(options[column], errors="coerce")

options["expiration"] = pd.to_datetime(options["expiration"], errors="coerce")
options["option_type"] = options["option_type"].str.lower()
options["lastTradeDate"] = pd.to_datetime(options.get("lastTradeDate"), errors="coerce", utc=True)

valid_quote = (options["bid"] >= 0) & (options["ask"] > 0) & (options["ask"] >= options["bid"])
options["mid_price"] = np.where(valid_quote, (options["bid"] + options["ask"]) / 2, np.nan)
options["market_price"] = options["mid_price"].where(options["mid_price"] > 0, options["lastPrice"])
options["relative_spread"] = (options["ask"] - options["bid"]) / options["mid_price"]

options["days_to_expiry"] = (options["expiration"] - today).dt.days
options["time_to_expiry"] = options["days_to_expiry"] / 365
options["moneyness"] = options["strike"] / spot
options["forward_price"] = spot * np.exp((RISK_FREE_RATE - dividend_yield) * options["time_to_expiry"])
options["log_moneyness"] = np.log(options["strike"] / options["forward_price"])

now_utc = pd.Timestamp.now(tz="UTC")
options["quote_age_days"] = (now_utc - options["lastTradeDate"]).dt.total_seconds() / 86400

good_rows = (
    options["option_type"].isin(["call", "put"])
    & options["market_price"].gt(0)
    & options["days_to_expiry"].between(MIN_DTE, MAX_DTE)
    & options["moneyness"].between(MIN_MONEYNESS, MAX_MONEYNESS)
    & (options["relative_spread"].le(MAX_RELATIVE_SPREAD) | options["relative_spread"].isna())
    & (options["quote_age_days"].le(MAX_QUOTE_AGE_DAYS) | options["quote_age_days"].isna())
)
options = options.loc[good_rows].copy()

print("Rows after basic cleaning:", len(options))

Rows after basic cleaning: 2389


In [8]:
def calculate_row_iv(row):
    solved = solve_iv(
        row["option_type"], row["market_price"], spot, row["strike"],
        row["time_to_expiry"], RISK_FREE_RATE, dividend_yield
    )

    if np.isfinite(solved) and 0.01 <= solved <= 3.0:
        return solved, "mid-price solver"

    vendor_iv = row["impliedVolatility"]
    if np.isfinite(vendor_iv) and 0.01 <= vendor_iv <= 3.0:
        return vendor_iv, "vendor fallback"

    return np.nan, "missing"

iv_results = options.apply(calculate_row_iv, axis=1)
options["implied_volatility"] = [result[0] for result in iv_results]
options["iv_source"] = [result[1] for result in iv_results]
options = options.dropna(subset=["implied_volatility"]).copy()


def calculate_row_greeks(row):
    return pd.Series(
        bsm_greeks(
            row["option_type"], spot, row["strike"], row["time_to_expiry"],
            RISK_FREE_RATE, dividend_yield, row["implied_volatility"]
        )
    )

greeks = options.apply(calculate_row_greeks, axis=1)
clean_options = pd.concat([options, greeks], axis=1).reset_index(drop=True)

clean_options["is_otm"] = (
    ((clean_options["option_type"] == "call") & (clean_options["strike"] >= clean_options["forward_price"]))
    | ((clean_options["option_type"] == "put") & (clean_options["strike"] < clean_options["forward_price"]))
)

clean_options["delta_exposure"] = clean_options["delta"] * spot * CONTRACT_MULTIPLIER * clean_options["openInterest"].fillna(0)
clean_options["gamma_exposure_1pct"] = clean_options["gamma"] * spot**2 * 0.01 * CONTRACT_MULTIPLIER * clean_options["openInterest"].fillna(0)
clean_options["vega_exposure_1vol"] = clean_options["vega"] * CONTRACT_MULTIPLIER * clean_options["openInterest"].fillna(0)
clean_options["theta_exposure_1day"] = clean_options["theta"] * CONTRACT_MULTIPLIER * clean_options["openInterest"].fillna(0)

print("Final clean option rows:", len(clean_options))
print("Expirations:", clean_options["expiration"].nunique())

Final clean option rows: 2389
Expirations: 9


## 5. Quick overview

Summary of the cleaned option data and key metrics.


In [9]:
clean_options["atm_distance"] = (clean_options["strike"] - clean_options["forward_price"]).abs()
atm_index = clean_options.groupby("expiration")["atm_distance"].idxmin()
atm_options = clean_options.loc[atm_index].sort_values("expiration").copy()

overview = pd.DataFrame({
    "Metric": [
        "Symbol", "Spot price", "Clean contracts", "Expirations",
        "Median relative spread", "IV solved from midpoint"
    ],
    "Value": [
        market["symbol"],
        f"${spot:,.2f}",
        f"{len(clean_options):,}",
        clean_options["expiration"].nunique(),
        f"{clean_options['relative_spread'].median():.2%}",
        f"{(clean_options['iv_source'] == 'mid-price solver').mean():.1%}",
    ],
})
display(overview)

expiration_summary = clean_options.groupby("expiration").agg(
    contracts=("strike", "count"),
    open_interest=("openInterest", "sum"),
    median_spread=("relative_spread", "median"),
    median_iv=("implied_volatility", "median"),
).reset_index()

display(expiration_summary)

,Metric,Value
0,Symbol,SPY
1,Spot price,$767.05
2,Clean contracts,"2,389"
3,Expirations,9
4,Median relative spread,3.72%
5,IV solved from midpoint,100.0%


,expiration,contracts,open_interest,median_spread,median_iv
0,2026-09-08,171,39864.0,0.064516,0.131659
1,2026-09-09,181,31702.0,0.045548,0.137629
2,2026-09-10,161,12704.0,0.045042,0.131882
3,2026-09-11,288,339605.0,0.054198,0.170574
4,2026-09-18,449,2900743.0,0.039216,0.226271
5,2026-09-25,247,113616.0,0.038557,0.137655
6,2026-09-30,479,830366.0,0.028113,0.181893
7,2026-10-02,241,67028.0,0.029346,0.134908
8,2026-10-09,172,6768.0,0.012988,0.132129


## 6. Volatility smile

Shows how implied volatility changes across strike prices for out-of-the-money options.


In [10]:
smile = clean_options.loc[clean_options["is_otm"]].copy()
smile["expiration_label"] = smile["expiration"].dt.strftime("%Y-%m-%d")
smile["plot_size"] = np.log1p(smile["openInterest"].fillna(0)) + 1

smile_fig = px.scatter(
    smile,
    x="moneyness",
    y="implied_volatility",
    color="expiration_label",
    symbol="option_type",
    size="plot_size",
    hover_data=["strike", "market_price", "openInterest", "days_to_expiry"],
    title=f"{market['symbol']}: OTM Implied-Volatility Smile",
    labels={
        "moneyness": "Strike / Spot",
        "implied_volatility": "Implied volatility",
        "expiration_label": "Expiration",
    },
)
smile_fig.add_vline(x=1.0, line_dash="dash")
smile_fig.update_yaxes(tickformat=".0%")
smile_fig.show()

## 7. ATM volatility term structure

Shows how near-ATM implied volatility changes across expiration dates, with recent realized volatility for comparison.


In [11]:
close = pd.to_numeric(market["history"]["Close"], errors="coerce").dropna()
log_returns = np.log(close / close.shift(1))

realized_vol = {}
for window in [20, 60, 120]:
    if len(log_returns.dropna()) >= window:
        realized_vol[window] = log_returns.tail(window).std() * math.sqrt(252)

term_fig = go.Figure()
term_fig.add_trace(go.Scatter(
    x=atm_options["days_to_expiry"],
    y=atm_options["implied_volatility"],
    mode="lines+markers",
    name="ATM implied volatility",
))

for window, value in realized_vol.items():
    term_fig.add_hline(
        y=value,
        line_dash="dot",
        annotation_text=f"{window}D realized: {value:.1%}",
    )

term_fig.update_layout(
    title=f"{market['symbol']}: ATM Volatility Term Structure",
    xaxis_title="Days to expiration",
    yaxis_title="Annualized volatility",
    yaxis_tickformat=".0%",
)
term_fig.show()

## 8. 3D implied-volatility surface

Shows implied volatility across strike and time to expiration. Interpolation fills the gaps between observed option values.


In [12]:
surface = clean_options.loc[clean_options["is_otm"]].copy()
surface = surface.sort_values("relative_spread").drop_duplicates(["expiration", "strike"])
surface = surface.dropna(subset=["moneyness", "days_to_expiry", "implied_volatility"])

x = surface["moneyness"].to_numpy()
y = surface["days_to_expiry"].to_numpy()
z = surface["implied_volatility"].to_numpy()

x_grid = np.linspace(np.quantile(x, 0.02), np.quantile(x, 0.98), 60)
y_grid = np.linspace(y.min(), y.max(), 60)
grid_x, grid_y = np.meshgrid(x_grid, y_grid)

try:
    grid_z = griddata((x, y), z, (grid_x, grid_y), method="linear")
    nearest_z = griddata((x, y), z, (grid_x, grid_y), method="nearest")
    grid_z = np.where(np.isfinite(grid_z), grid_z, nearest_z)
except Exception:
    grid_z = griddata((x, y), z, (grid_x, grid_y), method="nearest")

surface_fig = go.Figure()
surface_fig.add_trace(go.Surface(
    x=grid_x, y=grid_y, z=grid_z,
    name="Interpolated surface",
    colorbar={"title": "IV"},
))
surface_fig.add_trace(go.Scatter3d(
    x=x, y=y, z=z,
    mode="markers",
    marker={"size": 2, "color": "black"},
    name="Observed options",
))
surface_fig.update_layout(
    title=f"{market['symbol']}: Implied-Volatility Surface",
    scene={
        "xaxis_title": "Strike / Spot",
        "yaxis_title": "Days to expiration",
        "zaxis_title": "Implied volatility",
    },
    height=700,
)
surface_fig.show()

## 9. Liquidity

Compares relative bid-ask spreads across expirations and option types. Smaller spreads generally indicate better liquidity.


In [13]:
liquidity = clean_options.copy()
liquidity["expiration_label"] = liquidity["expiration"].dt.strftime("%Y-%m-%d")

liquidity = liquidity.groupby(["expiration_label", "option_type"]).agg(
    median_relative_spread=("relative_spread", "median"),
    total_volume=("volume", "sum"),
).reset_index()

liquidity_fig = px.bar(
    liquidity,
    x="expiration_label",
    y="median_relative_spread",
    color="option_type",
    barmode="group",
    hover_data=["total_volume"],
    title=f"{market['symbol']}: Median Relative Bid–Ask Spread",
    labels={
        "expiration_label": "Expiration",
        "median_relative_spread": "Median relative spread",
        "option_type": "Type",
    },
)
liquidity_fig.update_yaxes(tickformat=".0%")
liquidity_fig.show()

## 10. Greek exposure by expiration

Shows open-interest-weighted Delta, Gamma, Vega, and Theta by expiration. These are gross sensitivity measures, not long or short positioning.


In [14]:
greek_by_expiration = clean_options.copy()
greek_by_expiration["expiration_label"] = greek_by_expiration["expiration"].dt.strftime("%Y-%m-%d")

greek_by_expiration = greek_by_expiration.groupby("expiration_label").agg(
    delta=("delta_exposure", "sum"),
    gamma=("gamma_exposure_1pct", "sum"),
    vega=("vega_exposure_1vol", "sum"),
    theta=("theta_exposure_1day", "sum"),
).reset_index()

greek_fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Dollar Delta", "Dollar Gamma for 1% Spot Move", "Dollar Vega per Vol Point", "Dollar Theta per Day"),
)

greek_fig.add_bar(x=greek_by_expiration["expiration_label"], y=greek_by_expiration["delta"], row=1, col=1)
greek_fig.add_bar(x=greek_by_expiration["expiration_label"], y=greek_by_expiration["gamma"], row=1, col=2)
greek_fig.add_bar(x=greek_by_expiration["expiration_label"], y=greek_by_expiration["vega"], row=2, col=1)
greek_fig.add_bar(x=greek_by_expiration["expiration_label"], y=greek_by_expiration["theta"], row=2, col=2)

greek_fig.update_layout(
    title=f"{market['symbol']}: Open-Interest-Weighted Greek Concentration",
    height=700,
    showlegend=False,
)
greek_fig.show()

## 11. Historical realized volatility

Shows SPY price and realized volatility over time, with current near-30-day ATM implied volatility for comparison.


In [15]:
history = pd.DataFrame(index=close.index)
history["Close"] = close
history["20D realized"] = log_returns.rolling(20).std() * math.sqrt(252)
history["60D realized"] = log_returns.rolling(60).std() * math.sqrt(252)

history_fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=("Underlying Price", "Rolling Realized Volatility"),
)
history_fig.add_trace(go.Scatter(x=history.index, y=history["Close"], name="Close"), row=1, col=1)
history_fig.add_trace(go.Scatter(x=history.index, y=history["20D realized"], name="20D realized"), row=2, col=1)
history_fig.add_trace(go.Scatter(x=history.index, y=history["60D realized"], name="60D realized"), row=2, col=1)

near_30d_row = atm_options.iloc[(atm_options["days_to_expiry"] - 30).abs().argmin()]
current_atm_iv = float(near_30d_row["implied_volatility"])
history_fig.add_hline(
    y=current_atm_iv,
    line_dash="dash",
    annotation_text=f"Current near-30D ATM IV: {current_atm_iv:.1%}",
    row=2, col=1,
)

history_fig.update_yaxes(title_text="Price ($)", row=1, col=1)
history_fig.update_yaxes(title_text="Annualized volatility", tickformat=".0%", row=2, col=1)
history_fig.update_layout(title=f"{market['symbol']}: Price and Realized Volatility", height=700)
history_fig.show()

## 12. Export the main data tables

Save the cleaned option chain, expiration summary, and volatility-surface data.


In [16]:
OUTPUT_DIR.mkdir(exist_ok=True)

clean_options.to_csv(OUTPUT_DIR / "clean_option_chain.csv", index=False)
expiration_summary.to_csv(OUTPUT_DIR / "expiration_summary.csv", index=False)
surface.to_csv(OUTPUT_DIR / "volatility_surface_points.csv", index=False)

print("Saved files to:", OUTPUT_DIR.resolve())

Saved files to: /Users/apple/Downloads/options_dashboard_outputs
